# Diabetes readmission: data cleaning and insights

**Project question:** Using information available by hospital discharge, can we identify encounters recorded as readmitted within 30 days?

This notebook captures the cleaning decisions and checks we developed together. It is a starting point for the rest of the project: after running it, continue with cohort review, a patient-level split, preprocessing, and a baseline model. Each row is an encounter, so a patient can appear more than once.

**Source:** [UCI Diabetes 130-US Hospitals dataset](https://archive.ics.uci.edu/dataset/296/diabetes+130-us+hospitals+for+years+1999-2008), covering US hospital records from 1999–2008. Treat this as a historical learning project, not evidence of current performance in a London hospital. The original CSV is read but never modified.


## 1. Load and verify the source

In Colab, upload `diabetic_data.csv` before running this cell. The file-size and shape checks guard against the partial-upload problem we encountered earlier. `keep_default_na=False` preserves literal strings such as `None` in lab-result columns; the CSV uses `?` as an explicit missing marker.


In [ ]:
from pathlib import Path
import pandas as pd

candidates = [
    Path.cwd() / "diabetic_data.csv",
    Path.cwd().parent / "diabetic_data.csv",
]
csv_path = next((path for path in candidates if path.is_file()), None)
if csv_path is None:
    raise FileNotFoundError("Upload diabetic_data.csv or place it in the project folder.")

expected_bytes = 19_159_383
actual_bytes = csv_path.stat().st_size
print("Loaded file:", csv_path.resolve())
print("File size:", actual_bytes)
if actual_bytes != expected_bytes:
    raise ValueError("The CSV size differs from the verified source. Check that upload finished.")

fresh = pd.read_csv(csv_path, keep_default_na=False)
print("Shape:", fresh.shape)
if fresh.shape != (101_766, 50):
    raise ValueError("Expected the full 101,766-row, 50-column source dataset.")


## 2. Understand what one row represents

`encounter_id` identifies a hospital encounter. `patient_nbr` identifies a patient, who may have multiple encounters. Repeated patient IDs are expected; repeated encounter IDs or exact duplicate rows would need investigation.


In [ ]:
encounters_per_patient = fresh["patient_nbr"].value_counts()
print("Encounters:", len(fresh))
print("Unique patients:", fresh["patient_nbr"].nunique())
print("Patients with more than one encounter:", (encounters_per_patient > 1).sum())
print("Repeated encounter IDs:", fresh["encounter_id"].duplicated().sum())
print("Exact duplicate rows:", fresh.duplicated().sum())
print("Original readmission labels:")
print(fresh["readmitted"].value_counts().to_string())


**Insight:** The full file has 101,766 encounters from 71,518 unique patients; 16,773 patients have more than one encounter. We found no repeated encounter IDs and no exact duplicate rows. Keep repeat encounters, but later keep every encounter for a patient in the same train/test group.

The original target labels are `NO` (54,864), `>30` (35,545), and `<30` (11,357). We will define `<30` as the positive class. These are *recorded* outcomes in this dataset; `NO` should not be overstated as proof that a patient had no admission anywhere.


## 3. Audit missing markers and categories before changing data

Count `?` by column as a share of **encounters**. This tells us which fields may need special handling. Age and weight are recorded as intervals, not exact numeric measurements.


In [ ]:
question_mark_count = fresh.eq("?").sum()
missing_audit = pd.DataFrame({
    "question_marks": question_mark_count,
    "percent_of_encounters": 100 * question_mark_count / len(fresh),
}).query("question_marks > 0").sort_values("percent_of_encounters", ascending=False)
print(missing_audit.to_string())

for column in ["age", "weight", "gender"]:
    print(f"\n{column}")
    print(fresh[column].value_counts(dropna=False).to_string())


**Insight:** `weight` is missing for 98,569 encounters (96.86%). Other major `?` fields in the full data include `medical_specialty` (49,949) and `payer_code` (40,256). Age has ten string bands, such as `[70-80)`, and gender has three `Unknown/Invalid` records.

More encounters in an older age band do **not** mean that each older patient visits more often or that their readmission risk is higher. Those claims would need patient-level or outcome-rate calculations. A missing weight also does not make the entire encounter unusable.


## 4. Investigate weight without mistaking counts for risk

For each weight band, the readmission rate is the number of `<30` encounters in that band divided by **all encounters in the same band**. The mask created by `.eq(band)` marks matching rows. `?` is included so we can compare missing and recorded weight.


In [ ]:
readmitted_early = fresh["readmitted"].eq("<30")
weight_rows = []
for band in fresh["weight"].unique():
    in_band = fresh["weight"].eq(band)
    encounters = int(in_band.sum())
    early_readmissions = int((in_band & readmitted_early).sum())
    weight_rows.append({
        "weight_band": band,
        "encounters": encounters,
        "early_readmissions": early_readmissions,
        "readmission_rate": early_readmissions / encounters,
    })

weight_summary = pd.DataFrame(weight_rows).sort_values(
    "readmission_rate", ascending=False
)
print(weight_summary.to_string(index=False, formatters={
    "readmission_rate": lambda value: f"{value:.2%}"
}))


**Insight and decision:** `[0-25)` has the highest observed rate, 8/48 = 16.67%, but it contains only 48 encounters. The missing-weight group is 11,000/98,569 = 11.16%. Weight intervals are one encounter's recorded category; they do not show weight loss over time. Band size alone does not measure readmission risk, and these descriptive rates do not establish a causal effect.

For the **first model**, exclude `weight` as a feature because 96.86% of entries are missing. Keep the encounters. This is an initial modelling choice, not a claim that weight lacks clinical importance; a later experiment can revisit it.


## 5. Make a working copy and standardize missing values

Our first-pass decision is to remove the three encounters with `Unknown/Invalid` gender, then remove the `weight` column. Preserve `fresh` so the original data can always be rechecked. Replace literal `?` with pandas' missing marker, `pd.NA`; do not drop encounters simply because another field is unknown. Literal `None` in a lab-result field remains a meaningful category (test not recorded/performed) rather than being converted to `pd.NA`.


In [ ]:
valid_genders = ["Female", "Male"]
is_valid = fresh["gender"].isin(valid_genders)
print("Valid gender encounters:", int(is_valid.sum()))
print("Invalid gender encounters:", int((~is_valid).sum()))

filtered_fresh = fresh.loc[is_valid].copy()
filtered_fresh = filtered_fresh.drop(columns="weight")
cleaned = filtered_fresh.replace("?", pd.NA)

print("Working shape after gender and weight rules:", cleaned.shape)
assert cleaned.shape == (101_763, 49)
assert "weight" not in cleaned.columns
assert not cleaned.eq("?").any().any()
print("Remaining missing values:")
print(cleaned.isna().sum().loc[lambda counts: counts > 0].sort_values(ascending=False).to_string())


**Insight and decision:** After removing the three gender rows, 101,763 encounters remain. Six fields contain `pd.NA`: `medical_specialty` 49,947; `payer_code` 40,255; `race` 2,271; `diag_3` 1,423; `diag_2` 358; and `diag_1` 21. These differ slightly from full-file counts because three encounters were removed.

Keep these encounters. Deleting rows with missing `medical_specialty` alone would remove almost half the dataset. Missingness will be handled in preprocessing **after splitting**; whether the two high-missing fields help the model remains an experiment, not a settled fact.


## 6. Define the binary target

The source contains the exact string `"<30"` (no space). `.eq("<30")` checks category equality, then `.astype(int)` maps `True` to 1 and `False` to 0. This is **not** a numeric comparison of days.


In [ ]:
cleaned["readmitted_30"] = cleaned["readmitted"].eq("<30").astype(int)
print(cleaned["readmitted_30"].value_counts().sort_index().to_string())
print(f"Positive rate: {cleaned['readmitted_30'].mean():.2%}")
assert cleaned["readmitted_30"].value_counts().to_dict() == {0: 90_406, 1: 11_357}


**Insight:** The cleaned cohort has 90,406 negative and 11,357 positive encounters (11.16% positive). If the original target were truly missing, we would investigate why; we would not invent an outcome by filling it with 0. Encounters with unknown outcomes could not be used to train or score this supervised target.


## 7. Sanity-check numeric clinical counts

Minimum and maximum values detect impossible negatives or broken ranges. Quantiles describe how unusual a high value is; they do not prove an error. The source defines `num_medications` as distinct generic medication names administered during an encounter and `num_lab_procedures` as lab tests performed during an encounter ([original study, Table 1](https://pmc.ncbi.nlm.nih.gov/articles/PMC3996476/)).


In [ ]:
numeric_clinical_columns = [
    "time_in_hospital", "num_lab_procedures", "num_procedures",
    "num_medications", "number_outpatient", "number_emergency",
    "number_inpatient", "number_diagnoses",
]
print(cleaned[numeric_clinical_columns].agg(["min", "max"]).T.to_string())
assert (cleaned[numeric_clinical_columns].min() >= 0).all()
assert cleaned["time_in_hospital"].min() >= 1

print("\nMedication and lab-count quantiles:")
print(cleaned[["num_medications", "num_lab_procedures"]]
      .quantile([0.50, 0.95, 0.99]).to_string())

max_medications = cleaned["num_medications"].max()
extreme_medication_records = cleaned.loc[
    cleaned["num_medications"].eq(max_medications),
    ["encounter_id", "time_in_hospital", "num_medications",
     "num_lab_procedures", "number_diagnoses"],
]
print("\nEncounters at the maximum medication count:")
print(extreme_medication_records.to_string(index=False))


**Insight and decision:** `time_in_hospital` ranges from 1 to 14 days; medication counts from 1 to 81; lab-test counts from 1 to 132. The other checked count fields have no negative minima. The median/99th-percentile values are 15/43 for medications and 44/85 for lab tests, so the maxima are unusual.

Only one encounter has 81 recorded medications. It has a 10-day stay, 57 lab tests, and 6 diagnoses. This context does not verify that 81 is correct, but it does not establish an error either. **Flag and retain** the record for now; compare results with and without it later if it materially affects modelling.


## 8. Cleaning decision log

| Issue | Decision | Reason |
|---|---|---|
| Repeated `patient_nbr` | Keep encounters | One patient can have several valid hospital encounters. |
| Duplicate `encounter_id` or exact rows | None found | An encounter should represent one unique record. |
| `weight` missing in 96.86% | Exclude feature for first model; keep rows | Too little observed weight to rely on it initially. |
| Three `Unknown/Invalid` gender values | Remove those three encounters | Agreed first-pass rule; loss is about 0.003% of encounters. |
| `?` in six remaining columns | Convert to `pd.NA`; keep rows | Make missingness explicit without discarding substantial data. |
| Age intervals | Keep as categories | These are not exact numeric ages. |
| Extreme medication count (81) | Flag and retain | One unusual record is not proven incorrect. |
| Readmission labels | Encode `<30` as 1, other labels as 0 | Defines the discharge-time binary prediction question. |

These are reproducible decisions for this exercise, not universal clinical rules. Document any later change to the cohort or feature set and rerun the counts.


## 9. Set aside the target and identifiers

`X` contains **candidate predictors**, not the final approved model features. `readmitted` directly reveals the target, and `readmitted_30` *is* the target, so neither belongs in `X`. `encounter_id` has no clinical meaning as a feature. Keep `patient_nbr` separately to prevent the same patient appearing in both training and test sets. The `weight` column was already removed.


In [ ]:
X = cleaned.drop(columns=[
    "encounter_id", "patient_nbr", "readmitted", "readmitted_30"
]).copy()
y = cleaned["readmitted_30"].copy()
groups = cleaned["patient_nbr"].copy()

print("Candidate feature shape:", X.shape)
print("Target rows:", len(y), "group rows:", len(groups))
assert X.shape == (101_763, 46)
assert X.index.equals(y.index) and y.index.equals(groups.index)


## 10. Continue the project here

The basic cleaning audit is complete. **Before fitting a model**, review whether every encounter is eligible for the readmission question (for example, inspect `discharge_disposition_id` and its descriptions in `IDS_mapping.csv`; death or hospice dispositions may require an explicit cohort rule). We have **not** silently removed any such rows. Also review whether each candidate feature would be available at the chosen prediction time: discharge. Coded fields such as `admission_type_id`, `admission_source_id`, and `discharge_disposition_id` are categories, not numeric magnitudes.

Then split encounters **by `patient_nbr`**, keeping each patient's encounters entirely in train or test. Fit missing-value handling and categorical encoding on the training data only, transform the test data with those fitted rules, and compare a simple baseline with a first classifier. Because the positive class is about 11%, evaluate recall, precision, and calibration as well as any overall accuracy.

**Your next exercise:** inspect the discharge-disposition codes and propose an eligibility rule before making the patient-level split. If you change the cohort, recreate `X`, `y`, and `groups` from that cohort and update the decision log.


In [ ]:
# Continue here: inspect discharge disposition codes and their descriptions.
# Use IDS_mapping.csv in the project folder when you are ready.
